# Исключения в Python

Исключение - способ в Python сообщить, что код не может выполняться дальше без вмешательства в него программиста.

Некоторые исключения говорят нам, что код написан неверно - например `SyntaxError`:

In [2]:
print("Hi"+)

SyntaxError: invalid syntax (2625122622.py, line 1)

Рассмотрим случаи, когда код корректен, работает, но сталкивается с неверными данными или условиями.

## Встроенные исключения

В Python ряд операций невозможно совершить незаметно от пользователя, не получив ошибку. Делим на `0` - `ZeroDivisionError`:

In [3]:
1 / 0

ZeroDivisionError: division by zero

Используем в операции неподходящие типы (например, сумма числа и строки) - `TypeError`:

In [4]:
1 + "1"

TypeError: unsupported operand type(s) for +: 'int' and 'str'

Если с типом все в порядке, но проблема со значением - `ValueError`:

In [5]:
int("1")  # все ОК
int("abc")  # а тут неподходящее значение для преобразования в число

ValueError: invalid literal for int() with base 10: 'abc'

In [7]:
float("inf")
float("-inf")

-inf

Промах с индексом - `IndexError`:

In [8]:
arr = [1, 2, 3]
arr[4]

IndexError: list index out of range

А с ключом словаря - `KeyError`:

In [9]:
d = {"a": 1}
d["b"]

KeyError: 'b'

Если у типа данных нет такого метода - `AttributeError`:

In [10]:
arr.update(3)

AttributeError: 'list' object has no attribute 'update'

In [11]:
d.append("b", 10)

AttributeError: 'dict' object has no attribute 'append'

И много других встроенных ошибок.

Посмотрим на них поближе. Можем сами создать ошибку:

In [12]:
e = ValueError("Неподходящее значение")

Ошибка может принимать как параметр любоее сообщение (обычно в виде полезного текста, который подскажет, что именно не так).

У каждой ошибки - свой собственный тип:

In [14]:
print(type(e))
print(type(TypeError()))

<class 'ValueError'>
<class 'TypeError'>


Вызов ошибки - инструкция `raise`:

In [15]:
raise e

ValueError: Неподходящее значение

In [17]:
raise ValueError()

ValueError: 

Мы получаем сообщение типа:

```python
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
ValueError: Неподходящее значение
```

Это трейсбек - отчет об ошибке, который позволяет понять цепочку вызовов и локализовать в коде ошибку.

Видим также наше сообщение: `Неподходящее значение`.

Посмотрим пример посложнее:

In [18]:
def foo():
    bar()

def bar():
    baz()

def baz():
    raise ValueError

foo()

ValueError: 

Получаем цепочку типа:

```python
Traceback (most recent call last):
  File "<stdin>", line 1, in <module>
  File "<stdin>", line 2, in foo
  File "<stdin>", line 2, in bar
  File "<stdin>", line 2, in baz
ValueError
```

Читаем трейсбек снизу вверх - внизу место, где случилась ошибка - что именно сломалось. А дальше мы можем восстановить контекст - в каком случае сломалось.

## Обработка исключений

За обработку ошибок в Python отвечает блок `try...except...`:

In [22]:
try:
    1 / 0
except:
    print("Error")
print("Hi")

Error
Hi


Если операция может закончиться ошибкой - берем её в блок `try`, в случае ошибки сработает блок `except`.

Напишем функцию `get_key()`, которая пытается получить ключ словаря, а если его нет - вернуть `"Missing"`:

In [23]:
def get_key(d: dict[str, str], key: str) -> str:
    try:
        return d[key]
    except:
        print("Ошибка!")
        return "Missing"


print(get_key({"a": "b"}, "a"))
print(get_key({}, "a"))

b
Ошибка!
Missing


В блоке `except` мы можем обработать конкретную ошибку:

In [26]:
def divide():
    try:
        a = int(input("Введите делимое: "))
        b = int(input("Введите делитель: "))
        return a / b
    except ZeroDivisionError:
        print("Нельзя делить на 0!")

print(divide())

Введите делимое: 10
Введите делитель: abc


ValueError: invalid literal for int() with base 10: 'abc'

Если мы введем `0` в качестве делителя - мы обработаем ошибку `ZeroDivisionError`, но если `"abc"` - не обработаем `ValueError`.

Блоков `except` может быть несколько:

In [29]:
def divide():
    try:
        a = int(input("Введите делимое: "))
        b = int(input("Введите делитель: "))
        return a / b
    except ValueError:
        print("Введено некоректное число!")
    except ZeroDivisionError:
        print("Нельзя делить на 0!")
    print("Функция закончилась")

divide()

Введите делимое: 10
Введите делитель: 0
Нельзя делить на 0!
Функция закончилась


Лучше локализовать то место в коде, где именно может произойти ошибка - это упростит её поиск:

In [ ]:
def divide():
    try:
        a = int(input("Введите делимое: "))
    except ValueError:
        print("Введено некоректное делимое!")
        return

    try:
        b = int(input("Введите делитель: "))
    except ValueError:
        print("Введен некоректный делитель!")
        return

    try:
        return a / b
    except ZeroDivisionError:
        print("Нельзя делить на 0!")

divide()

В Python существует иерархия типов - наследование (о чем мы поговорим на следующих лекциях). Сейчас же отметим, что у ошибок тоже есть иерархия - мы можем отловить любые ошибки при помощи базовой ошибки `Exception`:

In [31]:
def divide():
    try:
        a = int(input("Введите делимое: "))
        b = int(input("Введите делитель: "))
        return a / b
    except Exception:
        print("Что-то не так!")

divide()

Введите делимое: abc
Что-то не так!


Мы можем сохранить ошибку в переменную - `except ... as ...`:

In [32]:
def divide():
    try:
        a = int(input("Введите делимое: "))
        b = int(input("Введите делитель: "))
        return a / b
    except Exception as e:
        print("Что-то не так!")
        print(e)
        print(type(e))

divide()

Введите делимое: 10
Введите делитель: 0
Что-то не так!
division by zero
<class 'ZeroDivisionError'>


При обработке ошибок можем использовать опциональный блок `else` - он вызовется, если ошибки не будет:

In [36]:
def foo():
    try:
        a = int(input("Введите число: "))
        print(a)
    except ValueError:
        print("Неподходящее число!")
        a = 2
    else:
        a = a ** 2
    print(a)

foo()

Введите число: 10
10
100


```
     try
    /  \
except else
    \  /
    code
```

Еще один опциональный блок - `finally`: он вызовется в любом случае, даже если мы не обработаем ошибку:

In [38]:
try:
    1/0
except ZeroDivisionError:
    print("Отловили ошибку")
finally:
    print("Этот блок тоже сработает")

Отловили ошибку
Этот блок тоже сработает


Блок `finally` чаще всего используется для освобождения ресурсов - например, если мы открыли файл, наткнулись на ошибку и хотим **в любом случае** закрыть файл:

In [40]:
%%writefile hello.txt

Hello!

Writing hello.txt


In [41]:
try:
    f = open("hello.txt")
    1 / 0
finally:
    print("Закрываем файл!")
    f.close()

Закрываем файл!


ZeroDivisionError: division by zero

In [42]:
def foo():
  try:
    return 10
  finally:
    print("Hi")

foo()

Hi


10

## Кастомные исключения

Системных исключений в большой программе недостаточно, поэтому обычно заводятся специальные кастомные исключения для ошибок, обычно завязанных на логику нашего приложения.

Такие исключения обычно наследуются от базового исключения `Exception`:

In [ ]:
class MyApplicationException(Exception):
    """Базовая ошибка нашего приложения"""

Обратите внимание - для имен классов исключений (и любых других классов) рекомендовано использовать не `snake_case`, а `PascalCase`.

Теперь можем создавать свои ошибки на базе `MyApplicationException`:

In [ ]:
class UserNotFoundException(MyApplicationException):
    pass

class NotEnoughMoneyException(MyApplicationException):
    pass

```
                 Exception
                     |
           MyApplicationException
                    /  \
UserNotFoundException  NotEnoughMoneyException
```

In [ ]:
def my_division(a, b):
  if b == 0:
      raise ZeroDivisionError()
  return a / b

Такие ошибки будут использоваться в конкретных функциях, а отлавливаться на уровни выше:

In [ ]:
def get_user(users: list, username: str) -> dict:
    """Найдет словарь с пользовательскими данными по имени"""
    for user in users:
        if user["username"] == username:
            return user
    raise UserNotFoundException(f"Пользователя с именем {username} не существует!")


def charge_money(amount: int, pays: list[int]) -> int:
    """Спишет суммы из списка pays со счета пользователя"""
    for pay in pays:
        amount -= pay
        if amount < 0:
            raise NotEnoughMoneyException("Недостаточно денег для списания!")
    return amount


def main():
    try:
        user = get_user([], "john")
        charge_money(user["amount"], [5, 4, 3, 2, 1])
    except MyApplicationException as e:  # Ловим любую нашу кастомную ошибку
        print(e)